In [0]:
from pyspark.sql.functions import trim, col, StringType
import pyspark.sql.functions as F
from pyspark.sql import functions as F


df = spark.table("dev_project.bronze.crm_prd_info")
col_name_map = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "cost",
    "prd_line": "product_line",
    "prd_start_dt":"start_date",
    "prd_end_dt": "end_date",
}

# trimming string valued column
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

df = (
    df.withColumn("prd_line", F.upper(trim(col("prd_line"))))
    .withColumn(
        "prd_line",
        F.when(F.col("prd_line").isNull(), "n/a").otherwise(F.upper(F.col("prd_line"))),
    )
    .withColumn(
        "prd_cost",
        F.when(F.col("prd_cost").isNull(), 0).otherwise(F.upper(F.col("prd_cost"))),
    )
    .withColumn("_old_start", F.col("prd_start_dt"))
    .withColumn("_old_end", F.col("prd_end_dt"))
    .withColumn(
        "prd_start_dt",
        F.when(F.col("_old_end") < F.col("_old_start"), F.col("_old_end")).otherwise(
            F.col("_old_start")
        ),
    )
    .withColumn(
        "prd_end_dt",
        F.when(F.col("_old_end") < F.col("_old_start"), F.col("_old_start")).otherwise(
            F.col("_old_end")
        ),
    )
    .drop("_old_start", "_old_end")
)

for old_name,new_name in col_name_map.items():
    df = df.withColumnRenamed(old_name, new_name)
df.display()

df.write.mode("overwrite").saveAsTable("dev_project.silver.crm_procuct")

In [0]:
df.filter(
    F.col("prd_end_dt") < F.col("prd_start_dt")
).count()